# Evaluación + videos del mejor modelo de CarRacing PPO

Esta notebook:

1. Reconstruye el mismo pipeline visual que usaste para entrenar.
2. Carga `best_model.zip`.
3. Evalúa el modelo en varios episodios.
4. Graba videos en la carpeta `videos/`.

> Antes de correr, ajustá `RUN_DIR` para que apunte a la carpeta de tu corrida.

In [1]:
%pip install "gymnasium[box2d]" stable-baselines3[extra] torch opencv-python pyyaml moviepy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import json
import numpy as np
import gymnasium as gym
import cv2

from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor, VecTransposeImage, VecVideoRecorder


In [3]:
# ========= CONFIGURACIÓN =========
RUN_DIR = Path("experiments/carracing_ppo/run_20260422_214043_seed0")

# Si querés evaluar el mejor modelo:
MODEL_PATH = RUN_DIR / "best_model" / "best_model.zip"

# Si no existe best_model.zip, podés usar el final:
# MODEL_PATH = RUN_DIR / "final_model.zip"

ENV_ID = "CarRacing-v3"
SEED = 0

# Debe coincidir con el entrenamiento
CONTINUOUS = True
GRAYSCALE = True
N_STACK = 4

# Evaluación
N_EVAL_EPISODES = 5
DETERMINISTIC = True

# Videos
VIDEO_DIR = RUN_DIR / "videos"
VIDEO_PREFIX = "best_model_eval"
VIDEO_EPISODE_LENGTH = 2000   # duración máxima por video
N_VIDEO_EPISODES = 3

assert RUN_DIR.exists(), f"No existe RUN_DIR: {RUN_DIR}"
assert MODEL_PATH.exists(), f"No existe MODEL_PATH: {MODEL_PATH}"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR.resolve())
print("MODEL_PATH:", MODEL_PATH.resolve())
print("VIDEO_DIR:", VIDEO_DIR.resolve())


RUN_DIR: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0
MODEL_PATH: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\best_model\best_model.zip
VIDEO_DIR: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\videos


In [4]:
# ========= WRAPPERS =========

class GrayscaleObservation(gym.ObservationWrapper):
    """Convierte RGB uint8 (H, W, 3) a grayscale uint8 (H, W, 1)."""

    def __init__(self, env: gym.Env):
        super().__init__(env)
        assert isinstance(env.observation_space, spaces.Box)
        obs_space = env.observation_space
        assert len(obs_space.shape) == 3 and obs_space.shape[-1] == 3, (
            f"Expected channel-last RGB image, got shape={obs_space.shape}"
        )
        h, w, _ = obs_space.shape
        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=(h, w, 1),
            dtype=np.uint8,
        )

    def observation(self, obs: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        return gray[..., None].astype(np.uint8)


In [5]:
# ========= FACTORY DEL ENTORNO =========

def build_single_env(seed: int, monitor_file: Path | None = None, render_mode: str | None = None):
    env = gym.make(
        ENV_ID,
        continuous=CONTINUOUS,
        render_mode=render_mode,
    )

    env.action_space.seed(seed)
    env.reset(seed=seed)

    if GRAYSCALE:
        env = GrayscaleObservation(env)

    if monitor_file is not None:
        env = Monitor(env, filename=str(monitor_file))
    else:
        env = Monitor(env)

    return env


def make_eval_vec_env(seed: int = 0, with_video: bool = False):
    render_mode = "rgb_array" if with_video else None

    def _init():
        return build_single_env(
            seed=seed,
            monitor_file=RUN_DIR / f"monitor_eval_notebook_seed{seed}.csv",
            render_mode=render_mode,
        )

    vec_env = DummyVecEnv([_init])
    vec_env = VecMonitor(vec_env, filename=str(RUN_DIR / "vec_monitor_eval_notebook.csv"))
    vec_env = VecFrameStack(vec_env, n_stack=N_STACK)
    vec_env = VecTransposeImage(vec_env)

    return vec_env


In [6]:
# ========= CARGA DEL MODELO =========

eval_env = make_eval_vec_env(seed=SEED, with_video=False)

print("Observation space:", eval_env.observation_space)
print("Action space:", eval_env.action_space)

model = PPO.load(MODEL_PATH, env=eval_env)

print("Modelo cargado correctamente.")


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Observation space: Box(0, 255, (4, 96, 96), uint8)
Action space: Box([-1.  0.  0.], 1.0, (3,), float32)
Modelo cargado correctamente.


In [7]:
# ========= EVALUACIÓN CUANTITATIVA =========

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=DETERMINISTIC,
    render=False,
    return_episode_rewards=False,
)

print(f"Mean reward over {N_EVAL_EPISODES} eval episodes: {mean_reward:.3f}")
print(f"Std reward: {std_reward:.3f}")


Mean reward over 5 eval episodes: 810.012
Std reward: 149.043


In [8]:
# ========= EVALUACIÓN EPISODIO POR EPISODIO =========

episode_rewards = []
episode_lengths = []

for ep in range(N_EVAL_EPISODES):
    obs = eval_env.reset()
    done = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while True:
        action, _states = model.predict(obs, deterministic=DETERMINISTIC)
        obs, rewards, dones, infos = eval_env.step(action)

        total_reward += float(rewards[0])
        steps += 1

        if dones[0]:
            break

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)
    print(f"Episode {ep+1}: reward={total_reward:.3f}, length={steps}")

print("\nResumen:")
print("Rewards:", episode_rewards)
print("Lengths:", episode_lengths)
print(f"Reward medio: {np.mean(episode_rewards):.3f}")
print(f"Reward std:   {np.std(episode_rewards):.3f}")


Episode 1: reward=763.158, length=1000
Episode 2: reward=506.742, length=1000
Episode 3: reward=873.333, length=1000
Episode 4: reward=882.759, length=1000
Episode 5: reward=804.918, length=1000

Resumen:
Rewards: [763.15790874511, 506.7415588274598, 873.3333368226886, 882.7586301118135, 804.9180319458246]
Lengths: [1000, 1000, 1000, 1000, 1000]
Reward medio: 766.182
Reward std:   137.033


In [9]:
# ========= GRABAR VIDEOS =========
# Esto crea archivos .mp4 dentro de RUN_DIR/videos

video_env = make_eval_vec_env(seed=SEED + 123, with_video=True)

video_env = VecVideoRecorder(
    video_env,
    video_folder=str(VIDEO_DIR),
    record_video_trigger=lambda step: step == 0,
    video_length=VIDEO_EPISODE_LENGTH,
    name_prefix=VIDEO_PREFIX,
)

video_model = PPO.load(MODEL_PATH, env=video_env)

for ep in range(N_VIDEO_EPISODES):
    obs = video_env.reset()
    total_reward = 0.0
    steps = 0

    while True:
        action, _ = video_model.predict(obs, deterministic=DETERMINISTIC)
        obs, rewards, dones, infos = video_env.step(action)

        total_reward += float(rewards[0])
        steps += 1

        if dones[0] or steps >= VIDEO_EPISODE_LENGTH:
            break

    print(f"Video episode {ep+1}: reward={total_reward:.3f}, length={steps}")

video_env.close()

print("\nVideos generados en:")
print(VIDEO_DIR.resolve())
print("\nArchivos:")
for p in sorted(VIDEO_DIR.glob("*.mp4")):
    print(" -", p.name)


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Video episode 1: reward=859.248, length=1000
Saving video to c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\videos\best_model_eval-step-0-to-step-2000.mp4
MoviePy - Building video c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\videos\best_model_eval-step-0-to-step-2000.mp4.
MoviePy - Writing video c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\videos\best_model_eval-step-0-to-step-2000.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\videos\best_model_eval-step-0-to-step-2000.mp4
Video episode 2: reward=862.848, length=1000
Video episode 3: reward=886.971, length=1000

Videos generados en:
C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\07_longrun_bestcfg\experiments\carracing_ppo\run_20260422_214043_seed0\videos

Archivos:
 - best_model_eval-step-0-to-step-1000.mp4
 - best_model_eval-step-0-to-step-2000.mp4


In [10]:
# ========= OPCIONAL: VISUALIZAR QUÉ HAY EN LA CARPETA DE LA CORRIDA =========

print("Contenido principal de RUN_DIR:")
for p in sorted(RUN_DIR.iterdir()):
    print(" -", p.name)

print("\nVideos encontrados:")
for p in sorted(VIDEO_DIR.glob("*")):
    print(" -", p.name)


Contenido principal de RUN_DIR:
 - best_model
 - checkpoints
 - config.json
 - config.yaml
 - eval
 - final_model.zip
 - monitor_eval_10000.csv.monitor.csv
 - monitor_eval_notebook_seed0.csv.monitor.csv
 - monitor_eval_notebook_seed123.csv.monitor.csv
 - monitor_train_0.csv.monitor.csv
 - monitor_train_1.csv.monitor.csv
 - monitor_train_2.csv.monitor.csv
 - monitor_train_3.csv.monitor.csv
 - monitor_train_4.csv.monitor.csv
 - monitor_train_5.csv.monitor.csv
 - monitor_train_6.csv.monitor.csv
 - monitor_train_7.csv.monitor.csv
 - tb
 - vec_monitor_eval.csv.monitor.csv
 - vec_monitor_eval_notebook.csv.monitor.csv
 - vec_monitor_train.csv.monitor.csv
 - videos

Videos encontrados:
 - best_model_eval-step-0-to-step-1000.mp4
 - best_model_eval-step-0-to-step-2000.mp4


In [11]:
import numpy as np
import pandas as pd

evaluation_path = RUN_DIR / "eval" / "evaluations.npz"
data = np.load(evaluation_path)

timesteps = data["timesteps"]
results = data["results"]
ep_lengths = data["ep_lengths"]

mean_rewards = results.mean(axis=1)
std_rewards = results.std(axis=1)
min_rewards = results.min(axis=1)
max_rewards = results.max(axis=1)

mean_lengths = ep_lengths.mean(axis=1)
std_lengths = ep_lengths.std(axis=1)

df = pd.DataFrame({
    "timesteps": timesteps,
    "reward_mean": mean_rewards,
    "reward_std": std_rewards,
    "reward_min": min_rewards,
    "reward_max": max_rewards,
    "ep_length_mean": mean_lengths,
    "ep_length_std": std_lengths,
})

print(df)

    timesteps  reward_mean  reward_std  reward_min  reward_max  \
0       25000  -110.017311    0.854465 -111.480789 -108.916313   
1       50000  -118.993240    4.054528 -123.838715 -113.960144   
2       75000   190.112030   95.587532   57.729919  307.403351   
3      100000   318.128265  150.637344  129.809921  582.277954   
4      125000   250.872223   67.116348  153.963791  322.708954   
5      150000   350.451233  160.789932   72.044357  520.563538   
6      175000   490.588135  254.229034   26.843750  728.182617   
7      200000   326.200165  110.512093  182.445831  513.380981   
8      225000   402.402740  241.226883   89.993362  691.103699   
9      250000   508.011169  185.645111  294.035736  814.286316   
10     275000   391.602417  234.102615   78.010147  782.158203   
11     300000   658.545532  121.261429  501.459839  771.334229   
12     325000   558.489380  277.087036   87.999008  872.227417   
13     350000   464.385834  224.023819  285.088440  843.070007   
14     375